# Day 48: Benchmark TTFT vs Throughput Across Batch Sizes

**Layer:** Implementation


## Concept Overview

TTFT scales linearly with batch size (more tokens to prefill), while throughput improves with batch size (amortized weight loading). The optimal batch size is determined by the latency SLO budget.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Benchmark TTFT vs Throughput Across Batch Sizes

TTFT and per-token throughput have opposite dependencies on batch size. This is the fundamental tradeoff in inference serving.


In [ ]:
import time, numpy as np, matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def simulate_prefill_decode_tradeoff(d_model=4096, num_layers=8, seq_len=512, max_batch=64):
    results = []
    W = torch.randn(d_model*4, d_model, device=device, dtype=torch.float16)
    W2 = torch.randn(d_model, d_model*4, device=device, dtype=torch.float16)

    for bs in [1, 2, 4, 8, 16, 32, 64]:
        if bs > max_batch: break
        x = torch.randn(bs, seq_len, d_model, device=device, dtype=torch.float16)
        # Warmup
        for _ in range(3):
            h = x.view(bs*seq_len, d_model) @ W.T
            h = F.gelu(h)
            _ = h @ W2.T
        if device=='cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(20):
            h = x.view(bs*seq_len, d_model) @ W.T
            h = F.gelu(h)
            out = h @ W2.T
        if device=='cuda': torch.cuda.synchronize()
        t_ms = (time.perf_counter()-t0)/20*1000
        ttft = t_ms  # time for full prefill
        tput_tokens_s = bs * seq_len / t_ms * 1000
        results.append({'bs': bs, 'ttft_ms': ttft, 'throughput': tput_tokens_s})
    return results

results = simulate_prefill_decode_tradeoff()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
batches = [r['bs'] for r in results]
ttfts = [r['ttft_ms'] for r in results]
thrpts = [r['throughput'] for r in results]
ax1.plot(batches, ttfts, 'r-o'); ax1.set_xlabel('Batch Size'); ax1.set_ylabel('TTFT (ms)')
ax1.set_title('TTFT vs Batch Size (grows linearly)'); ax1.grid(True)
ax2.plot(batches, thrpts, 'b-o'); ax2.set_xlabel('Batch Size'); ax2.set_ylabel('Throughput (tok/s)')
ax2.set_title('Throughput vs Batch Size'); ax2.grid(True)
plt.tight_layout(); plt.savefig('ttft_throughput.png', dpi=100); plt.show()
print(f'Batch=1:  TTFT={results[0]["ttft_ms"]:.1f}ms throughput={results[0]["throughput"]:.0f}tok/s')
print(f'Batch=64: TTFT={results[-1]["ttft_ms"]:.1f}ms throughput={results[-1]["throughput"]:.0f}tok/s')


## 2. Latency vs Throughput SLO Decision

Given an SLO budget, find the maximum batch size that keeps TTFT within bounds.


In [ ]:
def find_optimal_batch(results, ttft_slo_ms=200):
    eligible = [r for r in results if r['ttft_ms'] <= ttft_slo_ms]
    if not eligible: return results[0]
    return max(eligible, key=lambda r: r['throughput'])

for slo in [50, 100, 200, 500]:
    opt = find_optimal_batch(results, slo)
    print(f'SLO={slo}ms: optimal batch={opt["bs"]}, throughput={opt["throughput"]:.0f} tok/s')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- TTFT scales linearly with batch size (more tokens to prefill), while throughput improves with batch size (amortized weight loading).
- TTFT and throughput have exactly opposite batch size sensitivities — this is the defining tradeoff in inference serving and why continuous batching uses small dynamic batch sizes for interactive traffic..
- Day 48 implementation complete.
